In [15]:
import pyspark
import csv
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.functions import *

spark = SparkSession.builder.appName('BigD_OurD').getOrCreate()
sc = spark.sparkContext

In [16]:
########################
# Act 3: Partitioning  #
########################

# Initialization of the whole RDD
game_rdd = sc.textFile("vgsales.csv").map(lambda line: next(csv.reader([line], skipinitialspace=True)))

game_rdd.collect()
header = game_rdd.first()
game_rdd = game_rdd.filter(lambda x: x != header)

In [17]:
###################################
# Genre to Global Sale Comparison #
#    by Keiron Mirandilla         #
###################################

# Keep genre(5th) and global sales(last) as a tuple w/ Hash Partitioning
# Get sum of global sales and sort by highest
genre_sales = game_rdd.map(lambda x: (x[4], float(x[-1]))).partitionBy(4).persist()
summed_sales = genre_sales.reduceByKey(lambda a,b: a+b)

genres = summed_sales.collect()

In [18]:
print("Global Sales of Each Genre:")
for i in range(len(genres)):
    print(f"  {i+1}. {genres[i][0]} with {genres[i][1]:.2f} million sales.")

Global Sales of Each Genre:
  1. Role-Playing with 927.37 million sales.
  2. Shooter with 1037.37 million sales.
  3. Fighting with 448.91 million sales.
  4. Platform with 831.37 million sales.
  5. Puzzle with 244.95 million sales.
  6. Strategy with 175.12 million sales.
  7. Sports with 1330.93 million sales.
  8. Racing with 732.04 million sales.
  9. Action with 1751.18 million sales.
  10. Adventure with 239.04 million sales.
  11. Misc with 809.96 million sales.
  12. Simulation with 392.20 million sales.


In [20]:
# Nash A. Mapula

# Pipeline: Filter for 'Action' and clean the Year/Sales data
# use a try/except or a simple check to skip 'N/A' years
def clean_and_map(x):
    try:
        # Year is index 3, Global_Sales is index 10 (or -1)
        return (int(x[3]), float(x[10]))
    except:
        return None

# The Pipeline: Filter Action > Map to (Year, Sales) > Remove None values
yearly_action_sales = game_rdd.filter(lambda x: x[4] == "Action") \
                              .map(clean_and_map) \
                              .filter(lambda x: x is not None)

# Strategy: Range Partitioning
yas_partitioned = yearly_action_sales.sortByKey(numPartitions=8).persist()

# View the Result (Transformation Output)
# see the total sales for the first 5 years in our partitioned data
results = yas_partitioned.reduceByKey(lambda a, b: a + b).sortByKey()
results.take(5)

[(1980, 0.34), (1981, 14.84), (1982, 6.52), (1983, 2.86), (1984, 1.85)]

In [22]:
##############################
# Act 4: Data Preprocessing  #
#        and DataFrames      #
##############################

# Initialization and pre-processing of dataframe and schema
headers = [
    ("Show ID", StringType(), False),
    ("Type", StringType(), False),
    ("Title", StringType(), False),
    ("Director", StringType(), True),
    ("Cast", StringType(), True),
    ("Country", StringType(), True),
    ("Date Added", StringType(), True),
    ("Release Year", IntegerType(), False),
    ("Rating", StringType(), True),
    ("Duration", StringType(), True),
    ("Listed In", StringType(), False),
    ("Description", StringType(), False)
]

netflix_schema = StructType([
    StructField(*field) for field in headers
])
netflix_df = spark.read.csv("netflix_titles.csv", header=True, schema=netflix_schema)
#netflix_df = (netflix_df.withColumn("Date Added", to_date(trim(col("Date")), 'MMMM d, yyyy')).drop("Date"))

#netflix_df = (netflix_df
#    # 1. Trim and convert
#    .withColumn("Date Added", to_date(trim(col("Date")), 'MMMM d, yyyy'))
#    # 2. Drop the original column
#    .drop("Date")
#    # 3. Remove any rows where "Date Added" ended up as null
#    .filter(col("Date Added").isNotNull())
#)

kv_nulls = {key[0]: "N/A" for key in headers if key[2]}
netflix_df = netflix_df.fillna(kv_nulls)

In [23]:
#########################
# TV vs Movies Releases #
#   Through the Years   #
# ===================== #
# by Keiron Mirandilla  #
#########################

# Classify between TV Show and Movie
tv_df = netflix_df.where(netflix_df["Type"] == "TV Show")
movie_df = netflix_df.where(netflix_df["Type"] == "Movie")

### TV and Movie Count Logic
# Get every row where Realease Year has a value.
# Count each row by its year of release
# Rename `count` column into `x Releases`
###
m_count = movie_df.select("*").where(col("Release Year").isNotNull()) \
    .groupBy("Release Year").count().orderBy("count") \
    .withColumn("Movie Releases", col("count")).drop("count")
tv_count = tv_df.select("*").where(col("Release Year").isNotNull()) \
    .groupBy("Release Year").count().orderBy("count") \
    .withColumn("TV Releases", col("count")).drop("count")

# Join the 2 DFs to easily compare the two
releases_insight = m_count.join(tv_count, on="Release Year", how="inner")\
    .orderBy("Release Year", ascending=False)

In [27]:
# Insight: Top 10 Countries with the Most Content
#filter out "N/A" and nulls to get accurate locations
country_insight = netflix_df.filter((col("Country") != "N/A") & (col("Country").isNotNull())) \
    .groupBy("Country").count() \
    .orderBy(col("count").desc()) \
    .limit(10)

#Display the top 10 countries
print("Insight: Top 10 Countries with the Most Content")
country_insight.show()

#WRITE TO PARQUET
country_insight.write.mode("overwrite").parquet("/content/top_10_countries_insight.parquet")
print("Country insight successfully saved as Parquet!")

Insight: Top 10 Countries with the Most Content
+--------------+-----+
|       Country|count|
+--------------+-----+
| United States| 2817|
|         India|  972|
|United Kingdom|  419|
|         Japan|  245|
|   South Korea|  199|
|        Canada|  181|
|         Spain|  145|
|        France|  124|
|        Mexico|  110|
|         Egypt|  106|
+--------------+-----+

Country insight successfully saved as Parquet!


In [25]:
########################
# Save DFs to Parquets #
########################

releases_insight.write.format("parquet").save("/content/releases_insight")

In [28]:
import shutil

# This zips your parquet folder so it's easy to download
shutil.make_archive('top_10_countries_insight', 'zip', '/content/top_10_countries_insight.parquet')
print("Folder zipped and ready for download!")

Folder zipped and ready for download!


In [29]:
from google.colab import files

files.download('top_10_countries_insight.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>